# M7 — SQL Avançado


In [ ]:
# M7 — SQL Avançado | Subqueries, Particionamento e Visões

import pandas as pd
import sqlite3


# DADOS REAIS (cliente.csv e transacoes.csv da EBAC M7)

cliente = pd.DataFrame({
    'id_cliente':    [5, 1, 2, 4, 6],
    'nome':          ['jose','maria','valentina','joana','fernando'],
    'valor_compra':  [500.43, 150.70, 210.99, 1300.50, 86.55],
    'loja_cadastro': ['magalu','subway','postoshell','magalu','seveneleven']
})

transacoes = pd.DataFrame({
    'id_cliente':   [1, 2, 3, 1, 1, 3, 4, 5, 3],
    'id_transacao': [768805383,768805399,818770008,76856563,767573759,
                     818575758,764545534,76766789,8154567758],
    'valor_compra': [50.74,30.90,110.0,2000.90,15.70,2.99,50.74,10.0,1100.0],
    'id_loja':      ['magalu','giraffas','postoshell','magalu','subway',
                     'seveneleven','extra','subway','shopee']
})

# Tabela particionada = mesmos dados (simula as pastas id_loja=x do S3)
transacoes_part = transacoes.copy()

# Carrega no SQLite (simula o Athena)
con = sqlite3.connect(':memory:')
cliente.to_sql('cliente',         con, index=False, if_exists='replace')
transacoes.to_sql('transacoes',   con, index=False, if_exists='replace')
transacoes_part.to_sql('transacoes_part', con, index=False, if_exists='replace')

print("Tabelas carregadas!")
print(f"\ncliente ({len(cliente)} linhas):")
print(cliente.to_string(index=False))
print(f"\ntransacoes ({len(transacoes)} linhas):")
print(transacoes.to_string(index=False))


# QUERY 1 — Subquery
# Transações cujas lojas têm clientes com valor_compra > 160

print("\n" + "="*60)
print("QUERY 1 — Subquery: lojas de clientes com valor_compra > 160")
print("="*60)

query_1 = pd.read_sql_query("""
SELECT id_loja, id_cliente, id_transacao
FROM transacoes
WHERE id_loja IN (
    SELECT loja_cadastro
    FROM cliente
    WHERE valor_compra > 160
)
""", con)

print(query_1.to_string(index=False))
print(f"\nTotal: {len(query_1)} linhas")
query_1.to_csv('query_1.csv', index=False)
print("query_1.csv salvo")


# QUERY 2 — Particionamento
# SELECT na tabela particionada filtrando apenas magalu
# (No Athena escaneia menos dados pois só lê a pasta id_loja=magalu)

print("\n" + "="*60)
print("QUERY 2 — Particionamento: transacoes_part onde id_loja = magalu")
print("="*60)

query_2 = pd.read_sql_query("""
SELECT * FROM transacoes_part
WHERE id_loja = 'magalu'
""", con)

print(query_2.to_string(index=False))
print(f"\nTotal: {len(query_2)} linhas")
print("No Athena: tabela particionada escaneia MENOS dados que a tabela normal")
query_2.to_csv('query_2.csv', index=False)
print(" query_2.csv salvo")


# QUERY 3 — VIEW transacoesv100
# Cria view com transações de valor > 100 e renomeia id_loja

print("\n" + "="*60)
print("QUERY 3 — VIEW transacoesv100: valor_compra > 100")
print("="*60)

# Simula: CREATE VIEW transacoesv100 AS ...
# Depois: SELECT * FROM transacoesv100
query_3 = pd.read_sql_query("""
SELECT id_cliente, valor_compra, id_loja AS nome_loja
FROM transacoes
WHERE valor_compra > 100
""", con)

print(query_3.to_string(index=False))
print(f"\nTotal: {len(query_3)} linhas")
query_3.to_csv('query_3.csv', index=False)
print("query_3.csv salvo")


# QUERY 4 — VIEW clientevalor
# Top 2 transações por valor (ORDER BY DESC LIMIT 2)

print("\n" + "="*60)
print("QUERY 4 — VIEW clientevalor: top 2 por valor_compra")
print("="*60)

# Simula: CREATE VIEW clientevalor AS ...
# Depois: SELECT * FROM clientevalor
query_4 = pd.read_sql_query("""
SELECT id_cliente, valor_compra
FROM transacoes
ORDER BY valor_compra DESC
LIMIT 2
""", con)

print(query_4.to_string(index=False))
print(f"\nTotal: {len(query_4)} linhas")
query_4.to_csv('query_4.csv', index=False)
print("query_4.csv salvo")


# RESUMO FINAL

print("\n" + "="*60)
print("RESUMO — M7 SQL Avançado")
print("="*60)
for i, (q, desc) in enumerate(zip(
    [query_1, query_2, query_3, query_4],
    ['Subquery','Particionamento (magalu)','VIEW valor > 100','VIEW top 2 valor']
), 1):
    print(f"  query_{i}.csv → {len(q):>3} linhas  |  {desc}")


Tabelas carregadas!

cliente (5 linhas):
 id_cliente      nome  valor_compra loja_cadastro
          5      jose        500.43        magalu
          1     maria        150.70        subway
          2 valentina        210.99    postoshell
          4     joana       1300.50        magalu
          6  fernando         86.55   seveneleven

transacoes (9 linhas):
 id_cliente  id_transacao  valor_compra     id_loja
          1     768805383         50.74      magalu
          2     768805399         30.90    giraffas
          3     818770008        110.00  postoshell
          1      76856563       2000.90      magalu
          1     767573759         15.70      subway
          3     818575758          2.99 seveneleven
          4     764545534         50.74       extra
          5      76766789         10.00      subway
          3    8154567758       1100.00      shopee

QUERY 1 — Subquery: lojas de clientes com valor_compra > 160
   id_loja  id_cliente  id_transacao
    magalu      